# Region Sensitivity OpenRouter Evaluation — Prompt-matched Version

This notebook is designed to match the previous Gemini evaluation notebook as closely as possible.

It uses the same official QA JSON file:

```text
region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
```

It also constructs the model input using the same prompt structure as the Gemini notebook:

```python
full_prompt = (
    system_prompt
    + "\n\n"
    + question
    + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
    + "Do not explain your reasoning."
)
```

Important design choice:

- OpenRouter receives this `full_prompt` as a single user message.
- No extra OpenRouter system prompt is added.
- Gold fields are not sent to the model.
- The answer extraction function is copied from the Gemini notebook.

## 1. Imports and paths

In [1]:
import os
import re
import json
import time
import random
import hashlib
import getpass
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from tqdm import tqdm

In [2]:
# -----------------------------
# Paths
# -----------------------------

OUTPUT_DIR = Path("region_sensitivity_outputs")
EVAL_OUTPUT_DIR = Path("region_sensitivity_eval_outputs")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This should be the same official QA file used by the Gemini notebook.
OFFICIAL_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_1800_eval.json"

# Optional debug file if you have it.
DEBUG_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_debug_15.json"

# Use False for the official 1800-row evaluation.
# The notebook still has a 5-row dry run cell later, so you usually keep this as False.
USE_DEBUG = False

QA_PATH = DEBUG_QA_PATH if USE_DEBUG else OFFICIAL_QA_PATH

print("Official QA path:", OFFICIAL_QA_PATH)
print("Debug QA path:", DEBUG_QA_PATH)
print("Selected QA path:", QA_PATH)
print("Eval output dir:", EVAL_OUTPUT_DIR)

if not QA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find selected QA file: {QA_PATH.resolve()}\n\n"
        "This prompt-matched OpenRouter notebook expects the official JSON used by the Gemini notebook.\n"
        "Please make sure the file exists, or update OFFICIAL_QA_PATH to the correct absolute path."
    )

Official QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Debug QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_debug_15.json
Selected QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Eval output dir: region_sensitivity_eval_outputs


## 2. Load official QA JSON

This matches the Gemini notebook, which loads the official QA file from JSON.

In [3]:
# -----------------------------
# Load QA data
# -----------------------------

with open(QA_PATH, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

eval_df = pd.DataFrame(qa_data)

print("Loaded QA:", eval_df.shape)

if "task_type" in eval_df.columns:
    print("\nTask distribution:")
    print(eval_df["task_type"].value_counts())

if "answer" in eval_df.columns:
    print("\nAnswer distribution by task:")
    print(pd.crosstab(eval_df["task_type"], eval_df["answer"]))

print("\nColumns:")
print(eval_df.columns.tolist())

display(eval_df.head(2))

Loaded QA: (1800, 9)

Task distribution:
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64

Answer distribution by task:
answer                  A    B    C    D
task_type                               
management_priority   162  157  150  131
pairwise_comparison   293  307    0    0
top_sensitive_region  161  144  142  153

Columns:
['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']


,id,task_type,system_prompt,question,options,answer,gold_region,weather_condition,candidate_lgas
0,rs_context_v2_pairwise_comparison_0001,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nno_rain+strong_wind+very_h...,"{'A': 'Sydney', 'B': 'Leichhardt'}",B,Leichhardt,no_rain+strong_wind+very_humid+overcast,"[Sydney, Leichhardt]"
1,rs_context_v2_pairwise_comparison_0002,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nlight_rain+overcast\n\nCan...,"{'A': 'Auburn', 'B': 'Warringah'}",B,Warringah,light_rain+overcast,"[Auburn, Warringah]"


## 3. Check required fields

These are the same required fields as the Gemini notebook.

In [5]:
# -----------------------------
# Check evaluation fields
# -----------------------------

required_cols = [
    "id",
    "task_type",
    "system_prompt",
    "question",
    "answer",
    "gold_region"
]

missing_cols = [col for col in required_cols if col not in eval_df.columns]

if missing_cols:
    raise ValueError(
        f"Missing columns: {missing_cols}\n\n"
        "This means the selected QA file is not the same official JSON format used by the Gemini evaluation.\n"
        "Do not use the simplified CSV for prompt-matched multi-model evaluation."
    )

print("All required columns found.")
print("Columns in eval_df:")
print(eval_df.columns.tolist())

if not USE_DEBUG:
    assert len(eval_df) == 1800, (
        f"Official evaluation should use 1800 QA rows, but eval_df has {len(eval_df)} rows. "
        "Check that USE_DEBUG=False and the official JSON path is correct."
    )

All required columns found.
Columns in eval_df:
['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']


## 4. Build prompt exactly like the Gemini notebook

In [6]:
# -----------------------------
# Prompt construction
# -----------------------------

def build_full_prompt(system_prompt, question):
    """
    Build the exact same prompt string used in the Gemini notebook.

    Gemini notebook logic:
        full_prompt = (
            system_prompt
            + "\n\n"
            + question
            + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
            + "Do not explain your reasoning."
        )
    """
    return (
        str(system_prompt)
        + "\n\n"
        + str(question)
        + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
        + "Do not explain your reasoning."
    )


# Check the first prompt.
test_full_prompt = build_full_prompt(
    eval_df.iloc[0]["system_prompt"],
    eval_df.iloc[0]["question"]
)

print("First full prompt SHA256:")
print(hashlib.sha256(test_full_prompt.encode("utf-8")).hexdigest())

print("\nFirst full prompt preview:")
print(test_full_prompt[:3000])

print("\nGold fields are NOT included in the prompt.")
print("Gold answer for row 0, shown only for checking:", eval_df.iloc[0]["answer"])
print("Gold region for row 0, shown only for checking:", eval_df.iloc[0]["gold_region"])

First full prompt SHA256:
2a1d036b4a0d3aafbb726c5dac5fa74cfff7d0abc9781dc0a931799bd8679a81

First full prompt preview:
You are evaluating weather-sensitive urban traffic patterns using the provided regional context and traffic evidence. Use only the information provided in the question. Do not rely on external assumptions about the LGA names. A weather-sensitive traffic region is one whose observed traffic appears to deviate more strongly or more frequently from its expected traffic level under the given weather condition. Return only one option letter from the given options.

Weather condition:
no_rain+strong_wind+very_humid+overcast

Candidate LGAs:

A. Sydney
- Number of traffic stations: 12
- Average expected traffic volume: 1088.1
- Average observed traffic volume under this weather condition: 1052.1
- Observed traffic compared with typical level: lower than typical
- Dominant land-use type: residential
- POI density: high
- Significant traffic change frequency: high
- Typical tra

## 5. Answer extraction

This function is copied from the Gemini notebook to keep evaluation consistent.

In [7]:
# -----------------------------
# Extract option letter from model response
# -----------------------------

def extract_option_letter(response_text):
    """
    Extract A/B/C/D from model output.

    This function avoids taking the first option letter mentioned in the reasoning.
    It prioritises final-answer patterns such as:
    - "The final answer is C"
    - "Answer: C"
    - "\\boxed{C}"
    """

    if response_text is None:
        return None

    text = str(response_text).strip()
    upper_text = text.upper().strip()

    # Direct one-letter answer, allowing punctuation
    direct_match = re.match(r"^\s*([ABCD])\s*[\.\)]?\s*$", upper_text)
    if direct_match:
        return direct_match.group(1)

    # Normalise whitespace
    upper_text = re.sub(r"\s+", " ", upper_text)

    # LaTeX boxed answer, e.g. \boxed{C}
    boxed_match = re.search(r"\\?BOXED\{([ABCD])\}", upper_text)
    if boxed_match:
        return boxed_match.group(1)

    # Common final-answer patterns
    final_patterns = [
        r"THE FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER\s*[:\-]?\s*([ABCD])",
        r"THE ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER\s*[:\-]?\s*([ABCD])",
        r"OPTION\s*([ABCD])",
        r"CHOOSE\s*([ABCD])",
    ]

    for pattern in final_patterns:
        matches = re.findall(pattern, upper_text)
        if matches:
            return matches[-1]

    # Fallback: use the last standalone A/B/C/D, not the first.
    # This handles outputs that discuss all options before giving a final answer.
    all_matches = re.findall(r"\b([ABCD])\b", upper_text)

    if all_matches:
        return all_matches[-1]

    return None


# Quick test
test_outputs = [
    "A",
    "A.",
    "Option B",
    "The answer is C.",
    "D because ...",
    "The final answer is \\boxed{C}",
    "After comparing A, B, C, and D, the final answer is C.",
    "unknown"
]

for x in test_outputs:
    print(x, "->", extract_option_letter(x))

A -> A
A. -> A
Option B -> B
The answer is C. -> C
D because ... -> D
The final answer is \boxed{C} -> C
After comparing A, B, C, and D, the final answer is C. -> C
unknown -> None


## 6. OpenRouter configuration

Default model:

```text
meta-llama/llama-3.3-70b-instruct
```

This is the paid / normal endpoint. The free endpoint would have `:free` at the end, but this notebook defaults to the paid model for official evaluation.

In [8]:
# -----------------------------
# OpenRouter configuration
# -----------------------------

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL_PROVIDER = "openrouter"

# Paid / normal endpoint:
MODEL_NAME = "mistralai/mixtral-8x22b-instruct"

# Free endpoint, not recommended for official 1800-row evaluation:
# MODEL_NAME = "meta-llama/llama-3.3-70b-instruct:free"

MODEL_SLUG = MODEL_NAME.replace("/", "_").replace(":", "_")

# Use a new filename to avoid mixing with the previous non-prompt-matched dry run.
SAVE_PATH = EVAL_OUTPUT_DIR / f"results_{MODEL_SLUG}_context_v2_1800_promptmatched.csv"

# For deterministic classification-style evaluation across OpenRouter models.
# Note: Gemini notebook did not explicitly set decoding parameters.
# For OpenRouter models, keep these constant across all OpenRouter evaluations.
TEMPERATURE = 0
MAX_TOKENS = 20

# Delay to reduce rate-limit risk.
SLEEP_SECONDS = 2.0

print("Model provider:", MODEL_PROVIDER)
print("Model name:", MODEL_NAME)
print("Save path:", SAVE_PATH)

Model provider: openrouter
Model name: mistralai/mixtral-8x22b-instruct
Save path: region_sensitivity_eval_outputs/results_mistralai_mixtral-8x22b-instruct_context_v2_1800_promptmatched.csv


## 7. Set OpenRouter API key

Do not hard-code your key into the notebook.

In [9]:
# -----------------------------
# API key
# -----------------------------

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ").strip()

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is empty.")

print("OpenRouter API key loaded.")

Enter your OpenRouter API key:  ········


OpenRouter API key loaded.


## 8. OpenRouter call function

Important: no extra system message is added.

The model receives only the Gemini-style `full_prompt` as one user message.

In [10]:
# -----------------------------
# OpenRouter API call
# -----------------------------

def should_stop_run(status_code, response_text):
    """
    Detect errors where continuing would waste time or create many failed rows.
    """
    text = (response_text or "").lower()

    stop_phrases = [
        "invalid api key",
        "no auth credentials",
        "unauthorized",
        "forbidden",
        "insufficient credits",
        "credits",
        "quota",
        "rate limit",
        "too many requests",
        "payment required",
    ]

    if status_code in [401, 402, 403]:
        return True

    if any(phrase in text for phrase in stop_phrases):
        return True

    return False


def call_openrouter_model(full_prompt, model_name=MODEL_NAME, max_retries=3):
    """
    Call OpenRouter model and return raw text response.

    To match the Gemini prompt as closely as possible:
    - full_prompt is constructed using the same Gemini notebook formula.
    - full_prompt is sent as a single user message.
    - no extra OpenRouter system prompt is added.
    """

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://chat.openai.com/",
        "X-Title": "Region Sensitivity Prompt-Matched Evaluation",
    }

    payload = {
        "model": model_name,
        "messages": [
            {"role": "user", "content": full_prompt}
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                OPENROUTER_API_URL,
                headers=headers,
                json=payload,
                timeout=90,
            )

            if response.status_code == 200:
                data = response.json()
                return data["choices"][0]["message"]["content"]

            response_text = response.text
            last_error = f"HTTP {response.status_code}: {response_text[:1000]}"

            if should_stop_run(response.status_code, response_text):
                raise RuntimeError(
                    "STOP_RUN: OpenRouter quota/auth/credit/rate-limit problem. "
                    f"Details: {last_error}"
                )

            if response.status_code in [429, 500, 502, 503, 504]:
                wait_time = min(60, (2 ** attempt) + random.random())
                print(f"Transient error. Retry {attempt}/{max_retries} after {wait_time:.1f}s")
                time.sleep(wait_time)
                continue

            raise RuntimeError(last_error)

        except requests.exceptions.RequestException as e:
            last_error = repr(e)
            wait_time = min(60, (2 ** attempt) + random.random())
            print(f"Request exception. Retry {attempt}/{max_retries} after {wait_time:.1f}s")
            time.sleep(wait_time)

    raise RuntimeError(f"OpenRouter request failed after retries: {last_error}")

## 9. Single-call test

Run this before any batch evaluation.

In [11]:
# -----------------------------
# Single-call test
# -----------------------------

test_row = eval_df.iloc[0]
test_prompt = build_full_prompt(
    test_row["system_prompt"],
    test_row["question"]
)

print("Testing one OpenRouter call...")
print("ID:", test_row["id"])
print("Task:", test_row["task_type"])
print("Gold answer:", test_row["answer"])

raw_response = call_openrouter_model(test_prompt)
predicted_answer = extract_option_letter(raw_response)

print("\nRaw response:")
print(raw_response)

print("\nExtracted answer:")
print(predicted_answer)

print("\nIs correct:")
print(predicted_answer == test_row["answer"])

Testing one OpenRouter call...
ID: rs_context_v2_pairwise_comparison_0001
Task: pairwise_comparison
Gold answer: B

Raw response:
B

Extracted answer:
B

Is correct:
True


## 10. Resume helpers

Successful rows are skipped.

Previous error rows are intentionally rerun.

In [12]:
# -----------------------------
# Resume helpers
# -----------------------------

def is_successful_result(row):
    error_value = row.get("error", None)
    pred_value = row.get("predicted_answer", None)

    error_empty = pd.isna(error_value) or str(error_value).strip() == ""
    pred_valid = str(pred_value).strip().upper() in ["A", "B", "C", "D"]

    return error_empty and pred_valid


def load_existing_successes(save_path):
    """
    Load existing output and keep only successful rows.
    Error rows are dropped so they can be rerun.
    """
    if not Path(save_path).exists():
        print("No existing result file found. Starting fresh.")
        return pd.DataFrame(), set()

    existing_df = pd.read_csv(save_path)

    if len(existing_df) == 0:
        return existing_df, set()

    if "id" not in existing_df.columns:
        raise ValueError("Existing output file does not contain 'id'. Check file compatibility.")

    success_mask = existing_df.apply(is_successful_result, axis=1)
    success_df = existing_df[success_mask].copy()
    success_ids = set(success_df["id"].astype(str).tolist())

    print(f"Existing rows: {len(existing_df)}")
    print(f"Existing successful rows kept: {len(success_df)}")
    print(f"Previous error rows to rerun: {len(existing_df) - len(success_df)}")

    return success_df, success_ids


def save_results(results, save_path=SAVE_PATH):
    result_df = pd.DataFrame(results)

    if "id" in result_df.columns:
        result_df = result_df.sort_values(by="id").reset_index(drop=True)

    result_df.to_csv(save_path, index=False)
    return result_df

## 11. Main evaluation function

Use `run_limit=5` first.

Then use `run_limit=None` for the full remaining run.

In [13]:
# -----------------------------
# Main evaluation
# -----------------------------

def run_evaluation(run_limit=5, sleep_seconds=SLEEP_SECONDS):
    """
    Run OpenRouter evaluation with prompt-matched input and safe resume.

    Args:
        run_limit:
            5 for dry run.
            None for all remaining rows.
        sleep_seconds:
            Delay between successful requests.
    """

    success_df, success_ids = load_existing_successes(SAVE_PATH)

    work_df = eval_df.copy()
    work_df["id"] = work_df["id"].astype(str)

    rows_to_run = work_df[~work_df["id"].isin(success_ids)].copy()

    # Keep the original official QA order to match the Gemini evaluation order as closely as possible.
    if run_limit is not None:
        rows_to_run = rows_to_run.head(run_limit).copy()

    print("\nEvaluation setup")
    print("----------------")
    print("Model provider:", MODEL_PROVIDER)
    print("Model name:", MODEL_NAME)
    print("Total QA rows:", len(eval_df))
    print("Already successful:", len(success_ids))
    print("Rows to run now:", len(rows_to_run))
    print("Saving to:", SAVE_PATH)

    results = success_df.to_dict("records")

    if len(rows_to_run) == 0:
        print("No remaining rows to run.")
        return save_results(results, SAVE_PATH)

    for local_i, (_, row) in enumerate(rows_to_run.iterrows(), start=1):
        print(
            f"\n[{local_i}/{len(rows_to_run)}] "
            f"id={row['id']} task={row['task_type']} gold={row['answer']}"
        )

        raw_response = None
        predicted_answer = None
        error = None

        try:
            full_prompt = build_full_prompt(
                row["system_prompt"],
                row["question"]
            )

            raw_response = call_openrouter_model(
                full_prompt=full_prompt,
                model_name=MODEL_NAME,
                max_retries=3
            )

            predicted_answer = extract_option_letter(raw_response)

            result_row = {
                "id": row["id"],
                "task_type": row["task_type"],
                "weather_condition": row.get("weather_condition", None),
                "model_provider": MODEL_PROVIDER,
                "model_name": MODEL_NAME,
                "gold_answer": row["answer"],
                "gold_region": row["gold_region"],
                "raw_response": raw_response,
                "predicted_answer": predicted_answer,
                "is_correct": predicted_answer == row["answer"],
                "error": None,
                "started_at": datetime.now().isoformat(timespec="seconds"),
                "finished_at": datetime.now().isoformat(timespec="seconds"),
            }

            print("Raw response:", repr(raw_response))
            print("Parsed:", predicted_answer, "| Correct:", result_row["is_correct"])

        except Exception as e:
            error = str(e)

            result_row = {
                "id": row["id"],
                "task_type": row["task_type"],
                "weather_condition": row.get("weather_condition", None),
                "model_provider": MODEL_PROVIDER,
                "model_name": MODEL_NAME,
                "gold_answer": row["answer"],
                "gold_region": row["gold_region"],
                "raw_response": None,
                "predicted_answer": None,
                "is_correct": False,
                "error": error,
                "started_at": datetime.now().isoformat(timespec="seconds"),
                "finished_at": datetime.now().isoformat(timespec="seconds"),
            }

            print("Error:", error)

            if "STOP_RUN" in error:
                print("\nStopping evaluation to avoid writing many failed rows.")
                break

        results.append(result_row)

        # Save after every row to avoid losing progress.
        save_results(results, SAVE_PATH)

        print(f"Saved rows: {len(results)}")

        if error is None:
            time.sleep(sleep_seconds)

    results_df = save_results(results, SAVE_PATH)

    print("\nRun finished or stopped.")
    print("Saved results to:", SAVE_PATH)
    print("Total saved rows:", len(results_df))
    print("Number of errors:", results_df["error"].notna().sum())

    if len(results_df) > 0:
        print("\nOverall accuracy including errors as wrong:", results_df["is_correct"].mean())
        print("\nAccuracy by task:")
        print(results_df.groupby("task_type")["is_correct"].mean())

    return results_df

## 12. Dry run: 5 rows

Run this first.

If `predicted_answer` is parsed as A/B/C/D and there are no errors, continue to the full run.

In [14]:
dry_df = run_evaluation(run_limit=5, sleep_seconds=SLEEP_SECONDS)
display(dry_df.tail())

No existing result file found. Starting fresh.

Evaluation setup
----------------
Model provider: openrouter
Model name: mistralai/mixtral-8x22b-instruct
Total QA rows: 1800
Already successful: 0
Rows to run now: 5
Saving to: region_sensitivity_eval_outputs/results_mistralai_mixtral-8x22b-instruct_context_v2_1800_promptmatched.csv

[1/5] id=rs_context_v2_pairwise_comparison_0001 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 1

[2/5] id=rs_context_v2_pairwise_comparison_0002 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 2

[3/5] id=rs_context_v2_pairwise_comparison_0003 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 3

[4/5] id=rs_context_v2_pairwise_comparison_0004 task=pairwise_comparison gold=B
Raw response: 'A'
Parsed: A | Correct: False
Saved rows: 4

[5/5] id=rs_context_v2_pairwise_comparison_0005 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B |

,id,task_type,weather_condition,model_provider,model_name,gold_answer,gold_region,raw_response,predicted_answer,is_correct,error,started_at,finished_at
0,rs_context_v2_pairwise_comparison_0001,pairwise_comparison,no_rain+strong_wind+very_humid+overcast,openrouter,mistralai/mixtral-8x22b-instruct,B,Leichhardt,B,B,True,None,2026-07-10T14:53:23,2026-07-10T14:53:23
1,rs_context_v2_pairwise_comparison_0002,pairwise_comparison,light_rain+overcast,openrouter,mistralai/mixtral-8x22b-instruct,B,Warringah,B,B,True,None,2026-07-10T14:53:26,2026-07-10T14:53:26
2,rs_context_v2_pairwise_comparison_0003,pairwise_comparison,heavy_rain+very_humid+overcast,openrouter,mistralai/mixtral-8x22b-instruct,B,Parramatta,B,B,True,None,2026-07-10T14:53:28,2026-07-10T14:53:28
3,rs_context_v2_pairwise_comparison_0004,pairwise_comparison,no_rain+extreme_heat,openrouter,mistralai/mixtral-8x22b-instruct,B,Parramatta,A,A,False,None,2026-07-10T14:53:31,2026-07-10T14:53:31
4,rs_context_v2_pairwise_comparison_0005,pairwise_comparison,no_rain+very_humid,openrouter,mistralai/mixtral-8x22b-instruct,B,Warringah,B,B,True,None,2026-07-10T14:53:34,2026-07-10T14:53:34


## 13. Check dry-run summary

In [15]:
if SAVE_PATH.exists():
    check_df = pd.read_csv(SAVE_PATH)

    print("Rows saved:", len(check_df))
    print("Errors:", check_df["error"].notna().sum())

    print("\nPrediction distribution:")
    print(check_df["predicted_answer"].value_counts(dropna=False).sort_index())

    print("\nAccuracy so far:")
    print(check_df["is_correct"].mean())

    print("\nAccuracy by task:")
    print(check_df.groupby("task_type")["is_correct"].mean())

Rows saved: 5
Errors: 0

Prediction distribution:
predicted_answer
A    1
B    4
Name: count, dtype: int64

Accuracy so far:
0.8

Accuracy by task:
task_type
pairwise_comparison    0.8
Name: is_correct, dtype: float64


## 14. Full run: all remaining rows

Only run this after the 5-row dry run looks correct.

This will resume from the existing output file and skip successful rows.

In [16]:
# Full run.
# If the 5 dry-run rows were successful, this will skip them and run the remaining 1795 rows.
full_df = run_evaluation(run_limit=None, sleep_seconds=SLEEP_SECONDS)
display(full_df.tail())

Existing rows: 5
Existing successful rows kept: 5
Previous error rows to rerun: 0

Evaluation setup
----------------
Model provider: openrouter
Model name: mistralai/mixtral-8x22b-instruct
Total QA rows: 1800
Already successful: 5
Rows to run now: 1795
Saving to: region_sensitivity_eval_outputs/results_mistralai_mixtral-8x22b-instruct_context_v2_1800_promptmatched.csv

[1/1795] id=rs_context_v2_pairwise_comparison_0006 task=pairwise_comparison gold=A
Raw response: 'A'
Parsed: A | Correct: True
Saved rows: 6

[2/1795] id=rs_context_v2_pairwise_comparison_0007 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 7

[3/1795] id=rs_context_v2_pairwise_comparison_0008 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 8

[4/1795] id=rs_context_v2_pairwise_comparison_0009 task=pairwise_comparison gold=A
Raw response: 'A'
Parsed: A | Correct: True
Saved rows: 9

[5/1795] id=rs_context_v2_pairwise_comparison_0010 task=pair

,id,task_type,weather_condition,model_provider,model_name,gold_answer,gold_region,raw_response,predicted_answer,is_correct,error,started_at,finished_at
1795,rs_context_v2_top_sensitive_region_0596,top_sensitive_region,light_rain+very_humid+overcast,openrouter,mistralai/mixtral-8x22b-instruct,D,Ku-Ring-Gai,D,D,True,NaN,2026-07-10T15:48:13,2026-07-10T15:48:13
1796,rs_context_v2_top_sensitive_region_0597,top_sensitive_region,no_rain+very_humid,openrouter,mistralai/mixtral-8x22b-instruct,A,Sutherland,A,A,True,NaN,2026-07-10T15:48:16,2026-07-10T15:48:16
1797,rs_context_v2_top_sensitive_region_0598,top_sensitive_region,no_rain+strong_wind+overcast,openrouter,mistralai/mixtral-8x22b-instruct,A,Warringah,B,B,False,NaN,2026-07-10T15:48:18,2026-07-10T15:48:18
1798,rs_context_v2_top_sensitive_region_0599,top_sensitive_region,light_rain+very_humid,openrouter,mistralai/mixtral-8x22b-instruct,B,Lane Cove,B,B,True,NaN,2026-07-10T15:48:21,2026-07-10T15:48:21
1799,rs_context_v2_top_sensitive_region_0600,top_sensitive_region,moderate_rain+very_humid,openrouter,mistralai/mixtral-8x22b-instruct,D,Parramatta,C,C,False,NaN,2026-07-10T15:48:24,2026-07-10T15:48:24


In [3]:
# ============================================================
# Task 5 metric verification:
# Mixtral 8x22B Instruct
# ============================================================

import os
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score
)

# ------------------------------------------------------------
# 1. Directly specify the confirmed final result file
# ------------------------------------------------------------

file_path = (
    "region_sensitivity_eval_outputs/"
    "results_mistralai_mixtral-8x22b-instruct_"
    "context_v2_1800_promptmatched.csv"
)

print("Selected file:")
print(file_path)

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"The confirmed Mixtral result file was not found:\n{file_path}"
    )


# ------------------------------------------------------------
# 2. Load results
# ------------------------------------------------------------

df = pd.read_csv(file_path)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

if "model_name" in df.columns:
    print("\nModel names recorded in the CSV:")
    print(df["model_name"].value_counts(dropna=False))


# ------------------------------------------------------------
# 3. Confirm required columns
# ------------------------------------------------------------

required_columns = [
    "gold_answer",
    "predicted_answer",
    "task_type"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ------------------------------------------------------------
# 4. Standardise answers and task types
# ------------------------------------------------------------

def clean_answer(value):
    if pd.isna(value):
        return None

    answer = str(value).strip().upper()

    if answer in ["A", "B", "C", "D"]:
        return answer

    return None


df["gold_clean"] = df["gold_answer"].apply(clean_answer)
df["pred_clean"] = df["predicted_answer"].apply(clean_answer)

df["task_clean"] = (
    df["task_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

valid_df = df[
    df["gold_clean"].isin(["A", "B", "C", "D"])
    & df["pred_clean"].isin(["A", "B", "C", "D"])
].copy()


# ------------------------------------------------------------
# 5. Basic checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC CHECK")
print("=" * 70)

print("Total rows:", len(df))
print("Valid rows:", len(valid_df))
print("Invalid or empty predictions:", len(df) - len(valid_df))

if "error" in df.columns:
    error_count = (
        df["error"].notna()
        & df["error"].astype(str).str.strip().ne("")
    ).sum()

    print("Rows with error:", int(error_count))

print("\nTask counts:")
print(valid_df["task_clean"].value_counts())

if len(valid_df) == 0:
    raise ValueError(
        "The selected Mixtral CSV contains no valid A/B/C/D predictions."
    )


# ------------------------------------------------------------
# 6. Overall Accuracy and Macro-F1
# ------------------------------------------------------------

y_true = valid_df["gold_clean"]
y_pred = valid_df["pred_clean"]

overall_accuracy = accuracy_score(
    y_true,
    y_pred
)

overall_macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=["A", "B", "C", "D"],
    average="macro",
    zero_division=0
)

print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(f"Accuracy check: {overall_accuracy * 100:.2f}%")
print(f"Overall Macro-F1: {overall_macro_f1 * 100:.2f}%")


# ------------------------------------------------------------
# 7. Accuracy and Macro-F1 by question type
# ------------------------------------------------------------

task_settings = {
    "pairwise_comparison": ["A", "B"],
    "top_sensitive_region": ["A", "B", "C", "D"],
    "management_priority": ["A", "B", "C", "D"]
}

task_rows = []

for task_name, labels in task_settings.items():

    task_df = valid_df[
        valid_df["task_clean"] == task_name
    ].copy()

    if len(task_df) == 0:
        task_accuracy = None
        task_macro_f1 = None

    else:
        task_accuracy = accuracy_score(
            task_df["gold_clean"],
            task_df["pred_clean"]
        )

        task_macro_f1 = f1_score(
            task_df["gold_clean"],
            task_df["pred_clean"],
            labels=labels,
            average="macro",
            zero_division=0
        )

    task_rows.append({
        "Task Type": task_name,
        "Valid Rows": len(task_df),
        "Accuracy (%)": (
            round(task_accuracy * 100, 2)
            if task_accuracy is not None
            else None
        ),
        "Macro-F1 (%)": (
            round(task_macro_f1 * 100, 2)
            if task_macro_f1 is not None
            else None
        )
    })

task_table = pd.DataFrame(task_rows)

print("\n" + "=" * 70)
print("RESULTS BY QUESTION TYPE")
print("=" * 70)

display(task_table)


# ------------------------------------------------------------
# 8. Option-level Recall and F1
# ------------------------------------------------------------

options = ["A", "B", "C", "D"]

recall_values = recall_score(
    y_true,
    y_pred,
    labels=options,
    average=None,
    zero_division=0
)

f1_values = f1_score(
    y_true,
    y_pred,
    labels=options,
    average=None,
    zero_division=0
)

option_table = pd.DataFrame({
    "Option": options,
    "Recall (%)": [
        round(value * 100, 2)
        for value in recall_values
    ],
    "F1 (%)": [
        round(value * 100, 2)
        for value in f1_values
    ]
})

print("\n" + "=" * 70)
print("OPTION-LEVEL RECALL AND F1")
print("=" * 70)

display(option_table)


# ------------------------------------------------------------
# 9. Prediction distribution
# ------------------------------------------------------------

prediction_counts = (
    y_pred.value_counts()
    .reindex(options, fill_value=0)
)

distribution_table = pd.DataFrame({
    "Option": options,
    "Prediction Count": [
        int(prediction_counts[option])
        for option in options
    ],
    "Prediction Percentage (%)": [
        round(
            prediction_counts[option] / len(valid_df) * 100,
            2
        )
        for option in options
    ]
})

print("\n" + "=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

display(distribution_table)


# ------------------------------------------------------------
# 10. Consistency check against stored is_correct
# ------------------------------------------------------------

if "is_correct" in df.columns:
    stored_accuracy = pd.to_numeric(
        df.loc[valid_df.index, "is_correct"],
        errors="coerce"
    ).mean()

    print("\n" + "=" * 70)
    print("CONSISTENCY CHECK")
    print("=" * 70)

    print(
        f"Accuracy recalculated from answers: "
        f"{overall_accuracy * 100:.2f}%"
    )

    print(
        f"Accuracy from stored is_correct: "
        f"{stored_accuracy * 100:.2f}%"
    )

    print(
        f"Difference: "
        f"{abs(overall_accuracy - stored_accuracy) * 100:.6f} "
        "percentage points"
    )

Selected file:
region_sensitivity_eval_outputs/results_mistralai_mixtral-8x22b-instruct_context_v2_1800_promptmatched.csv

Shape:
(1800, 13)

Columns:
['id', 'task_type', 'weather_condition', 'model_provider', 'model_name', 'gold_answer', 'gold_region', 'raw_response', 'predicted_answer', 'is_correct', 'error', 'started_at', 'finished_at']

Model names recorded in the CSV:
model_name
mistralai/mixtral-8x22b-instruct    1800
Name: count, dtype: int64

BASIC CHECK
Total rows: 1800
Valid rows: 1800
Invalid or empty predictions: 0
Rows with error: 0

Task counts:
task_clean
management_priority     600
pairwise_comparison     600
top_sensitive_region    600
Name: count, dtype: int64

OVERALL RESULTS
Accuracy check: 62.00%
Overall Macro-F1: 60.14%

RESULTS BY QUESTION TYPE


,Task Type,Valid Rows,Accuracy (%),Macro-F1 (%)
0,pairwise_comparison,600,74.33,74.22
1,top_sensitive_region,600,53.17,52.92
2,management_priority,600,58.50,58.48



OPTION-LEVEL RECALL AND F1


,Option,Recall (%),F1 (%)
0,A,68.18,65.47
1,B,63.98,64.14
2,C,59.59,56.68
3,D,46.83,54.29



PREDICTION DISTRIBUTION


,Option,Prediction Count,Prediction Percentage (%)
0,A,667,37.06
1,B,605,33.61
2,C,322,17.89
3,D,206,11.44



CONSISTENCY CHECK
Accuracy recalculated from answers: 62.00%
Accuracy from stored is_correct: 62.00%
Difference: 0.000000 percentage points


## 15. Final summary

In [17]:
# -----------------------------
# Final summary
# -----------------------------

results_df = pd.read_csv(SAVE_PATH)

# Normalise error field.
results_df["error_clean"] = results_df["error"].fillna("").astype(str).str.strip()
valid_df = results_df[results_df["error_clean"] == ""].copy()

# Normalise correctness after loading CSV.
results_df["is_correct_bool"] = results_df["is_correct"].astype(str).str.lower().isin(["true", "1", "yes"])
valid_df["is_correct_bool"] = valid_df["is_correct"].astype(str).str.lower().isin(["true", "1", "yes"])

print("Model provider:", MODEL_PROVIDER)
print("Model name:", MODEL_NAME)
print("Output:", SAVE_PATH)
print("Total saved rows:", len(results_df))
print("Successful rows:", len(valid_df))
print("Error rows:", len(results_df) - len(valid_df))

print("\nAccuracy, valid-only:")
print(valid_df["is_correct_bool"].mean() if len(valid_df) > 0 else None)

print("\nAccuracy, errors as wrong:")
print(results_df["is_correct_bool"].mean() if len(results_df) > 0 else None)

print("\nTask-level summary:")
summary = (
    results_df
    .assign(success=lambda x: x["error_clean"] == "")
    .groupby("task_type")
    .agg(
        total_rows=("id", "count"),
        successful_rows=("success", "sum"),
        accuracy_errors_as_wrong=("is_correct_bool", "mean"),
    )
    .reset_index()
)

valid_summary = (
    valid_df
    .groupby("task_type")
    .agg(accuracy_valid_only=("is_correct_bool", "mean"))
    .reset_index()
)

summary = summary.merge(valid_summary, on="task_type", how="left")
display(summary)

Model provider: openrouter
Model name: mistralai/mixtral-8x22b-instruct
Output: region_sensitivity_eval_outputs/results_mistralai_mixtral-8x22b-instruct_context_v2_1800_promptmatched.csv
Total saved rows: 1800
Successful rows: 1800
Error rows: 0

Accuracy, valid-only:
0.62

Accuracy, errors as wrong:
0.62

Task-level summary:


,task_type,total_rows,successful_rows,accuracy_errors_as_wrong,accuracy_valid_only
0,management_priority,600,600,0.585000,0.585000
1,pairwise_comparison,600,600,0.743333,0.743333
2,top_sensitive_region,600,600,0.531667,0.531667


## 16. Save compact summary CSV

In [18]:
# -----------------------------
# Save compact model summary
# -----------------------------

summary_rows = []

results_df = pd.read_csv(SAVE_PATH)
results_df["error_clean"] = results_df["error"].fillna("").astype(str).str.strip()
results_df["success"] = results_df["error_clean"] == ""
results_df["is_correct_bool"] = results_df["is_correct"].astype(str).str.lower().isin(["true", "1", "yes"])

valid_df = results_df[results_df["success"]].copy()

summary_rows.append({
    "model_provider": MODEL_PROVIDER,
    "model_name": MODEL_NAME,
    "task_type": "overall",
    "total_rows": len(results_df),
    "successful_rows": int(results_df["success"].sum()),
    "error_rows": int((~results_df["success"]).sum()),
    "accuracy_valid_only": valid_df["is_correct_bool"].mean() if len(valid_df) > 0 else None,
    "accuracy_errors_as_wrong": results_df["is_correct_bool"].mean() if len(results_df) > 0 else None,
    "output_file": str(SAVE_PATH),
})

for task_value, sub in results_df.groupby("task_type"):
    sub_valid = sub[sub["success"]].copy()

    summary_rows.append({
        "model_provider": MODEL_PROVIDER,
        "model_name": MODEL_NAME,
        "task_type": task_value,
        "total_rows": len(sub),
        "successful_rows": int(sub["success"].sum()),
        "error_rows": int((~sub["success"]).sum()),
        "accuracy_valid_only": sub_valid["is_correct_bool"].mean() if len(sub_valid) > 0 else None,
        "accuracy_errors_as_wrong": sub["is_correct_bool"].mean() if len(sub) > 0 else None,
        "output_file": str(SAVE_PATH),
    })

model_summary_df = pd.DataFrame(summary_rows)

SUMMARY_PATH = (
    EVAL_OUTPUT_DIR
    / "model_accuracy_summary_openrouter_mixtral_8x22b_promptmatched.csv"
)

model_summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved summary:", SUMMARY_PATH)
print("Absolute path:", SUMMARY_PATH.resolve())
print("Exists:", SUMMARY_PATH.exists())

display(model_summary_df)

Saved summary: region_sensitivity_eval_outputs/model_accuracy_summary_openrouter_mixtral_8x22b_promptmatched.csv
Absolute path: /Users/tanghuiru/Desktop/labeled_data_v1/region_sensitivity_eval_outputs/model_accuracy_summary_openrouter_mixtral_8x22b_promptmatched.csv
Exists: True


,model_provider,model_name,task_type,total_rows,successful_rows,error_rows,accuracy_valid_only,accuracy_errors_as_wrong,output_file
0,openrouter,mistralai/mixtral-8x22b-instruct,overall,1800,1800,0,0.620000,0.620000,region_sensitivity_eval_outputs/results_mistra...
1,openrouter,mistralai/mixtral-8x22b-instruct,management_priority,600,600,0,0.585000,0.585000,region_sensitivity_eval_outputs/results_mistra...
2,openrouter,mistralai/mixtral-8x22b-instruct,pairwise_comparison,600,600,0,0.743333,0.743333,region_sensitivity_eval_outputs/results_mistra...
3,openrouter,mistralai/mixtral-8x22b-instruct,top_sensitive_region,600,600,0,0.531667,0.531667,region_sensitivity_eval_outputs/results_mistra...
